# k-Fold Cross Validation

## The Problem With a Single Train/Test Split

When you split your dataset once — say 75% training, 25% test — your performance estimate depends heavily on **which samples happened to land in the test set**. If the test set accidentally contains mostly easy examples, you will over-estimate your model's accuracy. If it contains hard outliers, you will under-estimate it.

This is not hypothetical. On a dataset of 400 samples with a 25% split, you have 100 test samples. The difference between 90% and 93% accuracy is just 3 data points. Your estimate is **high variance**.

---

## The Solution: k-Fold Cross Validation

k-Fold CV uses every sample for both training and testing, cycling through $k$ different splits:

```
k=5 example with 400 samples:

Fold 1:  [TEST  |  train | train | train | train]  → Accuracy: 91%
Fold 2:  [train | TEST   | train | train | train]  → Accuracy: 89%
Fold 3:  [train | train  | TEST  | train | train]  → Accuracy: 94%
Fold 4:  [train | train  | train | TEST  | train]  → Accuracy: 88%
Fold 5:  [train | train  | train | train | TEST ]  → Accuracy: 92%

Final estimate: mean = 90.8%, std = 2.2%
```

The model is trained $k$ times, each time on a different 80% of the data, and tested on the remaining 20%. The final performance estimate is the **average across all k folds**, with the **standard deviation** telling you how stable the model is.

---

## Why Both Mean and Standard Deviation Matter

| Result | What it tells you |
|--------|------------------|
| High mean, low std | Model generalises well and consistently |
| High mean, high std | Model is sensitive to which data it sees — might overfit on some folds |
| Low mean, low std | Underfitting consistently |
| Low mean, high std | Unstable and inaccurate — serious problem |

A model with 90% accuracy and 1% standard deviation is far more trustworthy than one with 92% accuracy and 8% standard deviation.

---

## Choosing k

| k value | Trade-off |
|---------|----------|
| k = 5 | Each fold trains on 80% of data. Fast. Slightly higher bias. |
| k = 10 | Standard choice. Good balance of bias and variance. |
| k = n (Leave-One-Out) | Lowest bias, very high compute cost, high variance on small datasets |

**Default: k = 10.** It is the most commonly used value in academic papers and competitions for small-to-medium datasets.

---

## What We Will Build

1. Train a Kernel SVM classifier on Social Network Ads data
2. Evaluate it with a single test split (the naive approach)
3. Apply 10-fold cross validation to get a more reliable estimate
4. Compare and see how the standard deviation reveals model stability

## Step 1: Import Libraries

| Library | Why we need it |
|---------|---------------|
| `numpy` | Array operations |
| `matplotlib` | Decision boundary visualisation |
| `pandas` | Loading the dataset |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Step 2: Load the Dataset

The Social Network Ads dataset has **400 users** with two features:

| Feature | Description |
|---------|-------------|
| `Age` | User age |
| `EstimatedSalary` | Estimated annual salary |

Target: `Purchased` (1 = bought the product, 0 = did not).

This is a small dataset by production standards — exactly where cross-validation matters most. With only 400 samples, a single random split produces a noisy performance estimate.

In [ ]:
dataset = pd.read_csv('Social_Network_Ads.csv')
X = dataset.iloc[:, :-1].values
y = dataset.iloc[:, -1].values

## Step 3: Initial Train/Test Split

We do an initial 75/25 split and train the model once. This gives us a baseline accuracy that we will later compare against the cross-validation result.

With 400 samples and a 25% test set, we have **100 test samples**. This means our accuracy estimate can swing by 3-4% depending purely on which 100 samples were randomly assigned to the test set — not because the model changed.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 0)

## Step 4: Feature Scaling

Kernel SVM computes distances between data points using the RBF (Radial Basis Function) kernel. If `EstimatedSalary` ranges from 15,000 to 150,000 and `Age` ranges from 18 to 60, the salary feature will dominate all distance calculations.

StandardScaler centres both features to mean 0, standard deviation 1 — so the model evaluates age and salary on equal footing.

**Always fit the scaler on training data only** — if you fit on the full dataset, test data statistics leak into the scaler, making your test evaluation optimistic.

In [ ]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

## Step 5: Train the Kernel SVM

We use an RBF (Radial Basis Function) kernel — this maps the 2D data into a higher-dimensional space where a non-linear decision boundary becomes linear.

The trained model gives us a single-split accuracy. But is this estimate reliable? That is what cross-validation will tell us.

In [ ]:
from sklearn.svm import SVC
classifier = SVC(kernel = 'rbf', random_state = 0)
classifier.fit(X_train, y_train)

## Step 6: Baseline Evaluation (Single Split)

The confusion matrix shows the breakdown of correct and incorrect predictions on our single test split:

```
[[TN  FP]     <- Predicted Not Purchased
 [FN  TP]]    <- Predicted Purchased
```

The accuracy here (~93%) is our **single-split estimate**. It looks good, but remember: it is based on just one random partitioning of 400 samples. In the next step, we will test whether this number holds up across multiple different splits.

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score
y_pred = classifier.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy_score(y_test, y_pred)

## Step 7: Apply 10-Fold Cross Validation

This is the key step. `cross_val_score` with `cv=10` does the following automatically:

1. Splits the **training set** into 10 equal folds
2. Trains the model 10 times, each time holding out one fold as a mini test set
3. Returns the accuracy from each fold as an array

We then report the **mean** (our best estimate of true accuracy) and **standard deviation** (how consistent the model is).

**Important:** Cross-validation is applied to the training set only (`X_train`, `y_train`). The test set we split off earlier is kept completely separate — it is our final held-out evaluation that only gets used once, at the very end.

**Interpreting the output:**

- If the CV mean (~90%) is close to the single-split accuracy (~93%), both estimates are consistent
- If they differ substantially, the single-split was lucky (or unlucky)
- A standard deviation below 5% is generally considered stable

In [ ]:
from sklearn.model_selection import cross_val_score
accuracies = cross_val_score(estimator = classifier, X = X_train, y = y_train, cv = 10)
print("Accuracy: {:.2f} %".format(accuracies.mean()*100))
print("Standard Deviation: {:.2f} %".format(accuracies.std()*100))

## Step 8: Visualise the Training Set Decision Boundary

This plot shows the RBF kernel SVM decision boundary on the training set. The non-linear curved boundary is only possible because of the kernel trick — mapping the original 2D features into a higher-dimensional space.

Training set visualisations look optimistic because the model has seen this data. The real test is the next plot.

In [ ]:
from matplotlib.colors import ListedColormap
X_set, y_set = X_train, y_train
X1, X2 = np.meshgrid(np.arange(start = X_set[:, 0].min() - 1, stop = X_set[:, 0].max() + 1, step = 0.01),
                     np.arange(start = X_set[:, 1].min() - 1, stop = X_set[:, 1].max() + 1, step = 0.01))
plt.contourf(X1, X2, classifier.predict(np.array([X1.ravel(), X2.ravel()]).T).reshape(X1.shape),
             alpha = 0.75, cmap = ListedColormap(('red', 'green')))
plt.xlim(X1.min(), X1.max())
plt.ylim(X2.min(), X2.max())
for i, j in enumerate(np.unique(y_set)):
    plt.scatter(X_set[y_set == j, 0], X_set[y_set == j, 1],
                c = ListedColormap(('red', 'green'))(i), label = j)
plt.title('Kernel SVM (Training set)')
plt.xlabel('Age')
plt.ylabel('Estimated Salary')
plt.legend()
plt.show()

## Step 9: Visualise the Test Set Decision Boundary

The test set points are data the model has never seen. Points landing in the wrong coloured region are misclassifications.

**Cross-validation vs this final test:**

- Cross-validation gave us a **reliable average estimate** of generalisation performance
- This plot shows performance on one specific held-out set

In production, you would report the cross-validation score as your model performance metric — not just the single test split — because it is based on more data and more independent evaluations.

In [ ]:
from matplotlib.colors import ListedColormap
X_set, y_set = X_test, y_test
X1, X2 = np.meshgrid(np.arange(start = X_set[:, 0].min() - 1, stop = X_set[:, 0].max() + 1, step = 0.01),
                     np.arange(start = X_set[:, 1].min() - 1, stop = X_set[:, 1].max() + 1, step = 0.01))
plt.contourf(X1, X2, classifier.predict(np.array([X1.ravel(), X2.ravel()]).T).reshape(X1.shape),
             alpha = 0.75, cmap = ListedColormap(('red', 'green')))
plt.xlim(X1.min(), X1.max())
plt.ylim(X2.min(), X2.max())
for i, j in enumerate(np.unique(y_set)):
    plt.scatter(X_set[y_set == j, 0], X_set[y_set == j, 1],
                c = ListedColormap(('red', 'green'))(i), label = j)
plt.title('Kernel SVM (Test set)')
plt.xlabel('Age')
plt.ylabel('Estimated Salary')
plt.legend()
plt.show()